In [0]:
from pyspark.sql import functions as F

SILVER = "workspace.s4lake_silver"
GOLD = "workspace.s4lake_gold"
DATA_CORTE = "2026-09-22"
DATA_SEPARACAO = "2025-12-31"
N_DIAS_ALVO = 30

fato_titulos = spark.table(f"{GOLD}.fato_titulos")

base = (
    fato_titulos
    .withColumn("dias_desde_vencimento",
                F.datediff(F.lit(DATA_CORTE).cast("date"), F.col("data_vencimento")))
    .withColumn("entra_treino",
                F.col("dias_desde_vencimento") > N_DIAS_ALVO)
    .withColumn("alvo",
                F.when(~F.col("entra_treino"), F.lit(None))
                 .when(F.col("dias_atraso") > N_DIAS_ALVO, F.lit(1))
                 .when(F.col("dias_atraso") <= N_DIAS_ALVO, F.lit(0)))
)

In [0]:
por_mes = (
    base
    .filter(F.col("entra_treino"))
    .withColumn("mes_emissao", F.date_format("data_base", "yyyy-MM"))
    .groupBy("mes_emissao")
    .agg(
        F.count(F.lit(1)).alias("qtd_titulos"),
        F.sum("alvo").alias("qtd_alvo_1"),
    )
    .withColumn("taxa_alvo",
                F.round(F.col("qtd_alvo_1") / F.col("qtd_titulos") * 100, 2))
    .orderBy("mes_emissao")
)

por_mes.show(30)

In [0]:
base_modelo = (
    base
    .filter(F.col("entra_treino"))
    .withColumn("conjunto",
                F.when(F.col("data_base") <= F.lit(DATA_SEPARACAO).cast("date"), F.lit("treino"))
                 .when(F.col("data_base") > F.lit(DATA_SEPARACAO).cast("date"), F.lit("teste")))
)

(
    base_modelo
    .groupBy("conjunto")
    .agg(
        F.count(F.lit(1)).alias("qtd_titulos"),
        F.sum("alvo").alias("qtd_alvo_1"),
    )
    .withColumn("taxa_alvo",
                F.round(F.col("qtd_alvo_1") / F.col("qtd_titulos") * 100, 2))
    .show()
)



In [0]:
# Passo 1: a dimensão de clientes, só com o que o modelo usa
clientes_dim = (
    spark.table(f"{SILVER}.clientes")
    .select("cod_cliente", "ramo_atividade", "uf")
)

# Passo 2: juntar com os títulos (títulos à esquerda: o grão é o título)
dados_modelo = base_modelo.join(clientes_dim, on="cod_cliente", how="left")

# Passo 3: criar a feature do mês
dados_modelo = dados_modelo.withColumn("mes_vencimento", F.month("data_vencimento"))

# Passo 4: ficar só com as colunas do modelo (protege contra vazamento)
dados_modelo = dados_modelo.select(
    "cod_cliente", "cod_fatura", # chave
    "valor", "prazo_dias", "mes_vencimento", "ramo_atividade", "uf", # features
    "alvo", "conjunto", # alvo e separação
)

# Validação
print(dados_modelo.count())
dados_modelo.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in dados_modelo.columns]
).show()

In [0]:
FEATURES_NUM = ["valor", ]
FEATURES_CAT = ["ramo_atividade", "uf", "mes_vencimento", "prazo_dias"]

dados_pd = dados_modelo.toPandas()
dados_pd["valor"] = dados_pd["valor"].astype(float)

treino = dados_pd[dados_pd["conjunto"] == "treino"]
teste = dados_pd[dados_pd["conjunto"] == "teste"]

X_treino = treino[FEATURES_NUM + FEATURES_CAT]
y_treino = treino["alvo"]
X_teste = teste[FEATURES_NUM + FEATURES_CAT]
y_teste = teste["alvo"]

print(X_treino.shape, y_treino.sum())
print(X_teste.shape, y_teste.sum())